# Classification master sheets

Joins the scored CSVs of one level (`LEVEL = "paragraph"` or `"sentence"`) into three training files under `data/<LEVEL>/master/`:

- `master-pairs-cumulative.csv` — each edit `v_k` against the original `v_0`. `delta_* = v_k − v_0`.
- `master-pairs-step.csv` — each edit `v_k` against the previous version `v_{k−1}`. `delta_* = v_k − v_{k−1}`. Depth 1 matches cumulative.
- `master-absolute.csv` — paper-style binary rows from consecutive step pairs. For each pair `⟨v_{k−1}, v_k⟩`, emit two rows with absolute metric values: older → `label=base`, newer → `label=edit`. Middle versions can appear twice with opposite labels. Columns `pair_older_depth` / `pair_newer_depth` record which step pair a row came from.

Both pair files share columns. `base_depth` says which version is on the left. Shuffle A/B later, in the training notebook, and split by article.

**Sentence level:** a link between two versions with `bleu_to_previous <= BLEU_THRESHOLD` (0.06, the same value as
data-preprocessing stage 5 uses for paragraph pairs) is not an edit of the same sentence, so the group is **split**
there: the versions after the weak link become a new group (its first version = depth 0). This way no step,
cumulative or absolute pair ever crosses a weak link. Split-off groups get the next free group numbers of the
article; untouched groups keep their number from `data/sentence/raw-csv`.

In [1]:
from pathlib import Path
import csv

LEVEL = "paragraph"  # "paragraph" or "sentence" - which data/<LEVEL>/features/ to read, data/<LEVEL>/master/ to write
features_dir = Path(f"../../data/{LEVEL}/features")  # per-article metric CSVs
master_dir = Path(f"../../data/{LEVEL}/master")  # master sheets are written here
master_dir.mkdir(parents=True, exist_ok=True)

# sentence level only: keep a link only if bleu_to_previous > BLEU_THRESHOLD, else split the group there.
# Paragraph pairs were already filtered at the same 0.06 in data-preprocessing stage 5, so no split here.
BLEU_THRESHOLD = 0.06 if LEVEL == "sentence" else None

metric_folders = [
    "kazmi-raw-csv",
    "kazmi-lexical-csv",
    "kazmi-pos-csv",
    "kazmi-syntactic-csv",
    "mdl-csv",
    "mean-surprisal-gpt2-csv",
    "mean-surprisal-srilm-csv",
]

id_columns = {"revision_group_no", "revision_depth", "text", "diff_url",
              # sentence csv only - context / traceability columns, not features
              "paragraph_group_no", "paragraph_depth", "prior", "post", "bleu_to_previous"}


def feature_columns(fieldnames):
    return [name for name in fieldnames if name not in id_columns]


def read_metric_csv(path):
    with path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        if not reader.fieldnames:
            return None
        features = feature_columns(reader.fieldnames)
        rows = {}
        for raw in reader:
            key = (raw["revision_group_no"], raw["revision_depth"])
            if key in rows:
                return "duplicate_key"
            rows[key] = raw
        return features, rows


def expected_features(stem):
    by_folder = {}
    ordered = []
    for folder in metric_folders:
        parsed = read_metric_csv(features_dir / folder / stem)
        if not isinstance(parsed, tuple):
            return None
        features, _rows = parsed
        by_folder[folder] = features
        ordered.extend(features)
    if len(ordered) != len(set(ordered)):
        return None
    return by_folder, ordered


def load_article(stem, expected_by_folder):
    loaded = []
    for folder in metric_folders:
        parsed = read_metric_csv(features_dir / folder / stem)
        if parsed == "duplicate_key":
            return "duplicate_key"
        if not isinstance(parsed, tuple):
            return "unreadable"
        features, rows = parsed
        if set(features) != set(expected_by_folder[folder]):
            return "feature_mismatch"
        loaded.append(rows)
    keys = set(loaded[0])
    for rows in loaded[1:]:
        if set(rows) != keys:
            return "key_mismatch"
    for rows in loaded[1:]:
        for key, raw in rows.items():
            if raw["text"] != loaded[0][key]["text"]:
                return "text_mismatch"
    groups = {}
    for key in keys:
        group_no, depth_text = key
        try:
            depth = int(depth_text)
        except ValueError:
            return "bad_depth"
        merged_features = {}
        for folder, rows in zip(metric_folders, loaded):
            source = rows[key]
            for name in expected_by_folder[folder]:
                merged_features[name] = source.get(name, "")
        groups.setdefault(group_no, {})[depth] = {
            "text": loaded[0][key]["text"],
            "diff_url": loaded[0][key].get("diff_url", ""),
            "bleu": loaded[0][key].get("bleu_to_previous", ""),
            "features": merged_features,
        }
    return groups


def split_weak_links(groups):
    """Sentence level: split a group wherever bleu_to_previous <= BLEU_THRESHOLD.
    Returns (groups, number of splits). Versions after a weak link become a new group with depths 0, 1, ..."""
    next_no = max((int(no) for no in groups), default=0) + 1
    result = {}
    splits = 0
    for group_no in sorted(groups, key=int):
        versions = groups[group_no]
        segment, current_no = {}, group_no
        for depth in sorted(versions):
            version = versions[depth]
            bleu = version["bleu"]
            if depth > 0 and bleu != "" and float(bleu) <= BLEU_THRESHOLD and segment:
                result[current_no] = segment  # close the group before the weak link
                current_no, next_no = str(next_no), next_no + 1
                segment = {}
                splits += 1
            segment[len(segment)] = version  # depth inside the (new) group
        result[current_no] = segment
    return result, splits


def format_value(value):
    if value is None or value == "":
        return ""
    return f"{float(value):.5f}"


def format_delta(edit_value, base_value):
    if edit_value is None or base_value is None or edit_value == "" or base_value == "":
        return ""
    return f"{float(edit_value) - float(base_value):.5f}"


def pair_row(article, group_no, base_depth, edit_depth, base, edit, feature_names):
    row = {
        "article": article,
        "revision_group_no": group_no,
        "base_depth": str(base_depth),
        "edit_depth": str(edit_depth),
        "diff_url": edit["diff_url"],
        "base_text": base["text"],
        "edit_text": edit["text"],
    }
    for name in feature_names:
        base_value = base["features"][name]
        edit_value = edit["features"][name]
        row[f"base_{name}"] = format_value(base_value)
        row[f"edit_{name}"] = format_value(edit_value)
        row[f"delta_{name}"] = format_delta(edit_value, base_value)
    return row


def absolute_row(
    article,
    group_no,
    depth,
    version,
    feature_names,
    label,
    pair_older_depth,
    pair_newer_depth,
):
    row = {
        "article": article,
        "revision_group_no": group_no,
        "revision_depth": str(depth),
        "pair_older_depth": str(pair_older_depth),
        "pair_newer_depth": str(pair_newer_depth),
        "label": label,
        "text": version["text"],
    }
    for name in feature_names:
        row[name] = format_value(version["features"][name])
    row["diff_url"] = version["diff_url"]
    return row


stem_sets = []
for folder in metric_folders:
    folder_dir = features_dir / folder
    stem_sets.append({path.name for path in folder_dir.glob("*.csv")})
common_stems = sorted(set.intersection(*stem_sets))
if not common_stems:
    raise RuntimeError(f"No article is present in all {len(metric_folders)} metric folders.")

schema = None
for stem in common_stems:
    schema = expected_features(stem)
    if schema is not None:
        break
if schema is None:
    raise RuntimeError("Could not read a feature header from the common articles.")
expected_by_folder, feature_names = schema

pair_fields = [
    "article",
    "revision_group_no",
    "base_depth",
    "edit_depth",
    "diff_url",
    "base_text",
    "edit_text",
]
for name in feature_names:
    pair_fields.extend([f"base_{name}", f"edit_{name}", f"delta_{name}"])

absolute_fields = [
    "article",
    "revision_group_no",
    "revision_depth",
    "pair_older_depth",
    "pair_newer_depth",
    "label",
    "text",
    *feature_names,
    "diff_url",
]

outputs = {
    "cumulative": master_dir / "master-pairs-cumulative.csv",
    "step": master_dir / "master-pairs-step.csv",
    "absolute": master_dir / "master-absolute.csv",
}
temps = {key: path.with_suffix(".csv.tmp") for key, path in outputs.items()}

counts = {
    "articles": 0,
    "skipped": 0,
    "cumulative": 0,
    "step": 0,
    "absolute": 0,
    "step_missing_prev": 0,
    "groups_split": 0,
}
skip_reasons = {}
completed = False
handles = {}

try:
    handles = {
        key: temp.open("w", encoding="utf-8", newline="")
        for key, temp in temps.items()
    }
    writers = {
        "cumulative": csv.DictWriter(handles["cumulative"], fieldnames=pair_fields),
        "step": csv.DictWriter(handles["step"], fieldnames=pair_fields),
        "absolute": csv.DictWriter(handles["absolute"], fieldnames=absolute_fields),
    }
    for writer in writers.values():
        writer.writeheader()

    for index, stem in enumerate(common_stems, start=1):
        article = stem[:-4]
        loaded = load_article(stem, expected_by_folder)
        if isinstance(loaded, str):
            counts["skipped"] += 1
            skip_reasons[loaded] = skip_reasons.get(loaded, 0) + 1
            print(f"skip {stem}: {loaded}")
            continue
        counts["articles"] += 1
        if BLEU_THRESHOLD is not None:
            loaded, n_splits = split_weak_links(loaded)
            counts["groups_split"] += n_splits
        for group_no in sorted(loaded, key=lambda value: int(value)):
            versions = loaded[group_no]
            if 0 not in versions:
                continue
            edit_depths = sorted(depth for depth in versions if depth >= 1)
            if not edit_depths:
                continue
            base = versions[0]
            for edit_depth in edit_depths:
                edit = versions[edit_depth]
                writers["cumulative"].writerow(
                    pair_row(article, group_no, 0, edit_depth, base, edit, feature_names)
                )
                counts["cumulative"] += 1
                prev_depth = edit_depth - 1
                if prev_depth not in versions:
                    counts["step_missing_prev"] += 1
                    continue
                older = versions[prev_depth]
                writers["step"].writerow(
                    pair_row(
                        article,
                        group_no,
                        prev_depth,
                        edit_depth,
                        older,
                        edit,
                        feature_names,
                    )
                )
                counts["step"] += 1
                # Paper binary: flatten consecutive pair into older=base, newer=edit rows.
                writers["absolute"].writerow(
                    absolute_row(
                        article,
                        group_no,
                        prev_depth,
                        older,
                        feature_names,
                        label="base",
                        pair_older_depth=prev_depth,
                        pair_newer_depth=edit_depth,
                    )
                )
                writers["absolute"].writerow(
                    absolute_row(
                        article,
                        group_no,
                        edit_depth,
                        edit,
                        feature_names,
                        label="edit",
                        pair_older_depth=prev_depth,
                        pair_newer_depth=edit_depth,
                    )
                )
                counts["absolute"] += 2
        if index % 500 == 0:
            print(f"joined {index} / {len(common_stems)}")
    completed = True
finally:
    for handle in handles.values():
        handle.close()
    if not completed:
        for temp in temps.values():
            temp.unlink(missing_ok=True)

for key, path in outputs.items():
    temps[key].replace(path)

print("articles_used", counts["articles"])
print("articles_skipped", counts["skipped"])
print("skip_reasons", skip_reasons)
print("cumulative_rows", counts["cumulative"])
print("step_rows", counts["step"])
print("absolute_rows", counts["absolute"])
print("step_pairs_skipped_missing_prev", counts["step_missing_prev"])
print("groups_split_at_weak_links", counts["groups_split"])
print("n_features", len(feature_names))
print("common_articles", len(common_stems))


joined 500 / 7519


joined 1000 / 7519


joined 1500 / 7519


joined 2000 / 7519


joined 2500 / 7519


joined 3000 / 7519


... (6 outputs left out)

joined 6500 / 7519


joined 7000 / 7519


joined 7500 / 7519


articles_used 7519
articles_skipped 0
skip_reasons {}
cumulative_rows 180536
step_rows 180536
absolute_rows 361072
step_pairs_skipped_missing_prev 0
n_features 104
common_articles 7519


In [2]:
# PARAGRAPH SPOT CHECK - one known paragraph article (skipped at sentence level)
if LEVEL == "paragraph":
    paragraph_dir = Path("../../data/paragraph/master")
    from pathlib import Path
    import csv

    article = "00s-Music-Quiz"
    group = "1"


    def collect(path):
        found = []
        with path.open("r", encoding="utf-8", newline="") as handle:
            for row in csv.DictReader(handle):
                if row["article"] == article and row["revision_group_no"] == group:
                    found.append(row)
                elif found:
                    break
        return found


    cumulative = collect(paragraph_dir / "master-pairs-cumulative.csv")
    step = collect(paragraph_dir / "master-pairs-step.csv")
    absolute = collect(paragraph_dir / "master-absolute.csv")

    assert len(cumulative) == 2
    assert {row["edit_depth"] for row in cumulative} == {"1", "2"}
    assert all(row["base_depth"] == "0" for row in cumulative)
    assert cumulative[0]["base_text"] == cumulative[1]["base_text"] == "For more information about..."

    step_by_depth = {row["edit_depth"]: row for row in step}
    assert set(step_by_depth) == {"1", "2"}
    assert step_by_depth["1"]["base_depth"] == "0"
    assert step_by_depth["2"]["base_depth"] == "1"
    assert step_by_depth["2"]["base_text"] == step_by_depth["1"]["edit_text"]

    delta_columns = [name for name in cumulative[0] if name.startswith("delta_")]
    cumulative_depth1 = next(row for row in cumulative if row["edit_depth"] == "1")
    assert all(cumulative_depth1[name] == step_by_depth["1"][name] for name in delta_columns)

    assert cumulative_depth1["delta_avg_sen_len"] == "8.00000"
    cumulative_depth2 = next(row for row in cumulative if row["edit_depth"] == "2")
    assert cumulative_depth2["delta_avg_sen_len"] == "7.00000"
    assert step_by_depth["2"]["delta_avg_sen_len"] == "-1.00000"

    # Absolute flattens consecutive step pairs: for depths 0,1,2 expect 4 rows.
    assert len(absolute) == 4
    assert [row["label"] for row in absolute] == ["base", "edit", "base", "edit"]
    assert [row["revision_depth"] for row in absolute] == ["0", "1", "1", "2"]
    assert [row["pair_older_depth"] for row in absolute] == ["0", "0", "1", "1"]
    assert [row["pair_newer_depth"] for row in absolute] == ["1", "1", "2", "2"]
    assert absolute[0]["text"] == cumulative[0]["base_text"]
    assert absolute[1]["text"] == absolute[2]["text"] == step_by_depth["1"]["edit_text"]
    assert sum(row["label"] == "base" for row in absolute) == 2
    assert sum(row["label"] == "edit" for row in absolute) == 2

    print("spot check ok")
    print("cumulative_group_rows", len(cumulative))
    print("step_group_rows", len(step))
    print("absolute_group_rows", len(absolute))


spot check ok
cumulative_group_rows 2
step_group_rows 2
absolute_group_rows 4


In [ ]:
# SENTENCE CHECK - no pair may cross a weak link (skipped at paragraph level)
if LEVEL == "sentence":
    import pandas as pd
    from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu

    smooth = SmoothingFunction().method1

    def bleu(base, revised):
        # same BLEU as data-preprocessing stage 5 and sentence-json-to-sentence-csv
        reference, hypothesis = str(base).lower().split(), str(revised).lower().split()
        if not reference or not hypothesis:
            return 0.0
        return sentence_bleu([reference], hypothesis, smoothing_function=smooth)

    step = pd.read_csv(master_dir / "master-pairs-step.csv", usecols=["base_text", "edit_text"], keep_default_na=False)
    sample = step.sample(n=min(20000, len(step)), random_state=0)
    weakest = min(bleu(b, e) for b, e in zip(sample["base_text"], sample["edit_text"]))
    print("step pairs:", len(step), "| lowest BLEU in a sample of", len(sample), "=", round(weakest, 4))
    assert weakest > BLEU_THRESHOLD, "a step pair crosses a weak link"

    absolute_rows = sum(1 for _ in open(master_dir / "master-absolute.csv", encoding="utf-8")) - 1
    assert absolute_rows == 2 * len(step), "absolute must have 2 rows per step pair"
    print("sentence check ok")